

| Files                        | Intended degradation  |
| ---------------------------- | --------------------- |
| `doc_01`, `doc_11`, `doc_21` | Gaussian noise        |
| `doc_02`, `doc_12`, `doc_22` | Salt-and-pepper noise |
| `doc_03`, `doc_13`, `doc_23` | Low contrast          |
| `doc_04`, `doc_14`, `doc_24` | Gaussian blur         |
| `doc_05`, `doc_15`, `doc_25` | Uneven lighting       |
| `doc_06`, `doc_16`           | JPEG artifacts        |
| `doc_07`, `doc_17`           | Sensor-like noise     |
| `doc_08`, `doc_18`           | Dark exposure         |
| `doc_09`, `doc_19`           | Perspective skew      |
| `doc_10`, `doc_20`           | Motion blur           |



In [3]:
import cv2
import numpy as np


def analyze_image(image_path):
    gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    if gray is None:
        raise ValueError(f"Cannot load image: {image_path}")

    img = gray.astype(np.float32)

    # ---------------------------------------
    # A. Brightness and contrast
    # ---------------------------------------
    p2, p98 = np.percentile(gray, [2, 98])
    brightness = float(np.mean(img))
    contrast_range = float(p98 - p2)

    # Estimate broad illumination variation.
    # Large variations may indicate uneven lighting,
    # but page content can also cause them.
    background = cv2.GaussianBlur(img, (0, 0), sigmaX=25)
    illumination_variation = float(
        np.percentile(background, 95)
        - np.percentile(background, 5)
    )

    # ---------------------------------------
    # B. Blur and edge detail
    # ---------------------------------------
    laplacian_variance = float(
        cv2.Laplacian(gray, cv2.CV_64F).var()
    )

    # ---------------------------------------
    # C. Salt-and-pepper-like outliers
    # ---------------------------------------
    median = cv2.medianBlur(gray, 3)

    difference = np.abs(
        gray.astype(np.int16) -
        median.astype(np.int16)
    )

    impulse_mask = (
        ((gray <= 5) | (gray >= 250))
        & (difference >= 50)
    )

    impulse_ratio = float(np.mean(impulse_mask))

    # ---------------------------------------
    # D. Estimate random noise
    # ---------------------------------------
    blurred = cv2.GaussianBlur(img, (3, 3), 0)
    residual = img - blurred

    gx = cv2.Sobel(img, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(img, cv2.CV_32F, 0, 1, ksize=3)
    gradient = cv2.magnitude(gx, gy)

    # Estimate noise in relatively smooth areas
    smooth_mask = gradient < 20

    if np.count_nonzero(smooth_mask) > 100:
        values = residual[smooth_mask]
    else:
        values = residual.ravel()

    med = np.median(values)
    noise_sigma = float(
        1.4826 * np.median(np.abs(values - med))
    )

    # ---------------------------------------
    # E. Report measurements
    # ---------------------------------------
    metrics = {
        "brightness_mean": round(brightness, 2),
        "contrast_percentile_range": round(contrast_range, 2),
        "illumination_variation": round(
            illumination_variation, 2
        ),
        "laplacian_variance": round(
            laplacian_variance, 2
        ),
        "impulse_outlier_ratio": round(impulse_ratio, 5),
        "estimated_noise_sigma": round(noise_sigma, 2),
    }

    # These are initial heuristic flags, not diagnoses.
    candidates = []

    if contrast_range < 100:
        candidates.append("Possible low contrast")

    if brightness < 100:
        candidates.append("Possible dark exposure")

    if impulse_ratio > 0.005:
        candidates.append("Possible salt-and-pepper noise")

    if noise_sigma > 5 and impulse_ratio <= 0.005:
        candidates.append("Possible Gaussian-like or sensor noise")

    if laplacian_variance < 100:
        candidates.append("Possible blur")

    if illumination_variation > 50:
        candidates.append(
            "Possible uneven illumination; inspect background"
        )

    # These need more specialized analysis.
    candidates.append(
        "JPEG artifacts: requires compression/blockiness analysis"
    )
    candidates.append(
        "Perspective skew: requires page-boundary/geometry analysis"
    )
    candidates.append(
        "Motion blur: requires directional blur analysis"
    )

    print("\n--- Image Analysis ---")

    for name, value in metrics.items():
        print(f"{name}: {value}")

    print("\n--- Candidate Degradations ---")

    for candidate in candidates:
        print("-", candidate)

    return {
        "metrics": metrics,
        "candidate_degradations": candidates,
    }

for i in range(1,26):
    
    if i<=9:
        
        result = analyze_image(
            f"Mini Project/noisy_images/doc_0{i}.png"
        )
    
    else:
        result = analyze_image(
        f"Mini Project/noisy_images/doc_{i}.png"
        )


--- Image Analysis ---
brightness_mean: 239.88
contrast_percentile_range: 95.0
illumination_variation: 14.9
laplacian_variance: 2377.05
impulse_outlier_ratio: 6e-05
estimated_noise_sigma: 6.67

--- Candidate Degradations ---
- Possible low contrast
- Possible Gaussian-like or sensor noise
- JPEG artifacts: requires compression/blockiness analysis
- Perspective skew: requires page-boundary/geometry analysis
- Motion blur: requires directional blur analysis

--- Image Analysis ---
brightness_mean: 237.89
contrast_percentile_range: 255.0
illumination_variation: 11.05
laplacian_variance: 30097.26
impulse_outlier_ratio: 0.02514
estimated_noise_sigma: 0.0

--- Candidate Degradations ---
- Possible salt-and-pepper noise
- JPEG artifacts: requires compression/blockiness analysis
- Perspective skew: requires page-boundary/geometry analysis
- Motion blur: requires directional blur analysis

--- Image Analysis ---
brightness_mean: 251.05
contrast_percentile_range: 22.0
illumination_variation: 4.